**Step 2 / 2**

**Script for analysing and plotting the final experiment data**

**NOTE:**
* Script 1 was executed first (`01-data-preprocessing.ipynb`)
* Check `requirements.txt` for the required packages if the automatic installation does not succeed
* Afterwards, free-text data in `results-02-cleaned.csv` was manually evaluated and then data points that were invalid due to the duration were discarded (> 400 s or < 10 s), e.g., participant with id 54 needed longer than 400 seconds at several tasks and 37+ min in total (dropped completely)
* This manually updated data was saved with the name `results-03-final.csv`

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import plot_likert as plik
import statsmodels.api as stm

pd.options.mode.chained_assignment = None  # default='warn'
# Change the second argument to 'None', to show always the complete dataframes
pd.set_option('display.max_rows', 50)

# Design settings for all boxplots
medianprops = dict(linestyle="-", linewidth=2.0, color="black")
flierprops = dict(marker="D", markerfacecolor="black",
                  markersize=5, markeredgecolor="none")

# util function to convert the experience levels to integers
def convert_experience(experience_string):
    match experience_string:
        case "Novice (1)":
            return 1
        case "Beginner (2)":
            return 2
        case "Intermediate (3)":
            return 3
        case "Advanced (4)":
            return 4
        case "Expert (5)":
            return 5

In [ ]:
normalizedData = pd.read_csv('results-03-final.csv', delimiter = ',')

### Participant Demographics

In [ ]:
# Aggregate year ranges together into bins (0-1,1-2,2-5,5-10,10-15,15+ years)
binnedYearsOfExp = pd.cut(normalizedData['DemYearsOE'], bins=[0, 1, 2, 5, 10, 15, 40], include_lowest=True)

# Draw the plot
yearsExpPlot = binnedYearsOfExp.value_counts().sort_index().plot(kind = 'bar', color = "g", rot = 0, figsize = (6,4), fontsize = 10)

# Change x-axis labels with standard ranges to custom labels
yearsExpPlot.set_xticks(np.arange(6), ["0-1", "1-2", "2-5", "5-10", "10-15", "15+"])
yearsExpPlot.grid(False)
yearsExpPlot.set_title('Years of Experience in IT', fontsize = 20)
yearsExpPlot.set_xlabel('Years', fontsize = 12)
yearsExpPlot.set_ylabel('Number of Participants', fontsize = 12)

print(" Median(Years of Experience):", np.median(normalizedData['DemYearsOE']), "\n Mean(Years of Experience):", np.mean(normalizedData['DemYearsOE']))
print(" 97% Quantile:", np.quantile(normalizedData['DemYearsOE'], q= 0.97))

In [ ]:
pltAcInd = normalizedData['DemAcInd'].value_counts().sort_index().plot(kind = 'pie', ylabel='', autopct='%1.1f%%', fontsize = 10)
pltAcInd.set_title('Professional Background', fontsize = 20)

In [ ]:
# Aggregate the two roles 'Manager' and 'DevOps Engineer' to 'Other' since they each only represent 1-2% of the total population
normalizedData['DemBackground'] = normalizedData['DemBackground'].replace({'Manager':'Other'})
normalizedData['DemBackground'] = normalizedData['DemBackground'].replace({'DevOps Engineer':'Other'})
normalizedData['DemBackground'] = pd.Categorical(normalizedData['DemBackground'], ["Student", "Software Engineer", "Researcher", "Software Architect", "Consultant", "Other"])
pltMainRole = normalizedData['DemBackground'].value_counts().sort_index().plot(kind = 'pie', ylabel='', autopct='%1.1f%%', fontsize = 10)
pltMainRole.set_title('Main Role', fontsize = 20)

In [ ]:
# Comparison between sequences ("randnumber")
demographics = normalizedData[["randnumber", "DemYearsOE", "DemAcInd", "DemBackground", "DemExperience[SQ001]",
                               "DemExperience[SQ002]", "DemExperience[SQ003]"]]
demographics["isStudent"] = demographics["DemBackground"] == "Student"
demographics["isIndustry"] = demographics["DemAcInd"] == "Industry"

demographics["DemExperience[patterns]"] = demographics["DemExperience[SQ001]"].apply(
    convert_experience)
demographics["DemExperience[APIs]"] = demographics["DemExperience[SQ002]"].apply(
    convert_experience)
demographics["DemExperience[MAPs]"] = demographics["DemExperience[SQ003]"].apply(
    convert_experience)

print(demographics.agg({
    "DemExperience[patterns]": ["mean", "median"],
    "DemExperience[APIs]": ["mean", "median"],
    "DemExperience[MAPs]": ["mean", "median"]
}))

demographics.groupby("randnumber").agg({
    "randnumber": ["count"],
    "DemYearsOE": ["mean", "median"],
    "isStudent": ["sum"],
    "isIndustry": ["sum"],
    "DemExperience[patterns]": ["mean", "median"],
    "DemExperience[APIs]": ["mean", "median"],
    "DemExperience[MAPs]": ["mean", "median"]
})

In [ ]:
print("Self-assessment experience in the area 'Software Design Patterns':")
normalizedData['DemExperience[SQ001]'] = pd.Categorical(normalizedData['DemExperience[SQ001]'], ["Novice (1)", "Beginner (2)", "Intermediate (3)", "Advanced (4)", "Expert (5)"])
normalizedData['DemExperience[SQ001]'].value_counts().sort_index()

In [ ]:
print("Self-assessment experience in the area 'HTTP APIs':")
normalizedData['DemExperience[SQ002]'] = pd.Categorical(normalizedData['DemExperience[SQ002]'], ["Novice (1)", "Beginner (2)", "Intermediate (3)", "Advanced (4)", "Expert (5)"])
normalizedData['DemExperience[SQ002]'].value_counts().sort_index()

In [ ]:
print("Self-assessment experience in the area 'API Patterns for Microservices':")
normalizedData['DemExperience[SQ003]'] = pd.Categorical(normalizedData['DemExperience[SQ003]'], ["Novice (1)", "Beginner (2)", "Intermediate (3)", "Advanced (4)", "Expert (5)"])
normalizedData['DemExperience[SQ003]'].value_counts().sort_index()

#### Plot of a 5-point Likert Scale
*Source: https://github.com/nmalkin/plot-likert*

In [ ]:
# Combine the three self-assessment series (SQ001,SQ002 and SQ003)
likert_series = {"\t Software Design Patterns": normalizedData['DemExperience[SQ001]'], "HTTP APIs": normalizedData[
    'DemExperience[SQ002]'], "API Patterns for \n Microservices": normalizedData['DemExperience[SQ003]']}
likert_data = pd.concat(likert_series, axis=1)
# Custom scale labes
likert_scale = {'Novice (1)': 'Novice (1)', 'Beginner (2)': 'Beginner (2)',
                'Intermediate (3)': 'Intermediate (3)', 'Advanced (4)': 'Advanced (4)', 'Expert (5)': 'Expert (5)'}
# Plot Likert
ax = plik.plot_likert(likert_data, likert_scale, plot_percentage=True, figsize=(8, 5), colors=plik.colors.default_with_darker_neutral)

for bars, color in zip(ax.containers[1:], ["white"] * 5):
    ax.bar_label(bars, label_type='center', fmt='%.1f %%', color=color)

### Task Responses

#### Calculation of TAU (Timed Actual Understandability)

`TAU (for a given task) = correctness * (1 - {duration[current participant] / max_duration[all participants]})`

##### Error Report

In [ ]:
# Get the max duration out of 2 columns (P and N version for a task), for all participants
maximum_ER1 = pd.concat([normalizedData['ER1PTime'],normalizedData['ER1NTime']]).max()
maximum_ER2 = pd.concat([normalizedData['ER2PTime'],normalizedData['ER2NTime']]).max()
TAU_ER1P = normalizedData['ER1P'] * (1 - (normalizedData['ER1PTime'] / maximum_ER1))
TAU_ER1N = normalizedData['ER1N'] * (1 - (normalizedData['ER1NTime'] / maximum_ER1))
TAU_ER2P = normalizedData['ER2P'] * (1 - (normalizedData['ER2PTime'] / maximum_ER2))
TAU_ER2N = normalizedData['ER2N'] * (1 - (normalizedData['ER2NTime'] / maximum_ER2))

print(maximum_ER1)
print(maximum_ER2)

print('TAU(ER1P): \n', TAU_ER1P,  '\n The median of ER1P: ', TAU_ER1P.median(), '\n The mean of ER1P: ', TAU_ER1P.mean())
print('TAU(ER1N): \n', TAU_ER1N, '\n The median of ER1N: ', TAU_ER1N.median(), '\n The mean of ER1N: ', TAU_ER1N.mean())

print('TAU(ER2P): \n', TAU_ER2P,  '\n The median of ER2P: ', TAU_ER2P.median(), '\n The mean of ER2P: ', TAU_ER2P.mean())
print('TAU(ER2N): \n', TAU_ER2N, '\n The median of ER2N: ', TAU_ER2N.median(), '\n The mean of ER2N: ', TAU_ER2N.mean())

##### Parameter Tree

In [ ]:
maximum_PT1 = pd.concat([normalizedData['PT1PTime'],normalizedData['PT1NTime']]).max()
maximum_PT2 = pd.concat([normalizedData['PT2PTime'],normalizedData['PT2NTime']]).max()
TAU_PT1P = normalizedData['PT1P'] * (1 - (normalizedData['PT1PTime'] / maximum_PT1))
TAU_PT1N = normalizedData['PT1N'] * (1 - (normalizedData['PT1NTime'] / maximum_PT1))
TAU_PT2P = normalizedData['PT2P'] * (1 - (normalizedData['PT2PTime'] / maximum_PT2))
TAU_PT2N = normalizedData['PT2N'] * (1 - (normalizedData['PT2NTime'] / maximum_PT2))

print(maximum_PT1)
print(maximum_PT2)

print('TAU(PT1P): \n', TAU_PT1P,  '\n The median of PT1P: ', TAU_PT1P.median(), '\n The mean of PT1P: ', TAU_PT1P.mean())
print('TAU(PT1N): \n', TAU_PT1N,  '\n The median of PT1N: ', TAU_PT1N.median(), '\n The mean of PT1N: ', TAU_PT1N.mean())

print('TAU(PT2P): \n', TAU_PT2P,  '\n The median of PT2P: ', TAU_PT2P.median(), '\n The mean of PT2P: ', TAU_PT2P.mean())
print('TAU(PT2N): \n', TAU_PT2N,  '\n The median of PT2N: ', TAU_PT2N.median(), '\n The mean of PT2N: ', TAU_PT2N.mean())

##### Wish List

In [ ]:
maximum_WL1 = pd.concat([normalizedData['WL1PTime'],normalizedData['WL1NTime']]).max()
maximum_WL2 = pd.concat([normalizedData['WL2PTime'],normalizedData['WL2NTime']]).max()
TAU_WL1P = normalizedData['WL1P'] * (1 - (normalizedData['WL1PTime'] / maximum_WL1))
TAU_WL1N = normalizedData['WL1N'] * (1 - (normalizedData['WL1NTime'] / maximum_WL1))
TAU_WL2P = normalizedData['WL2P'] * (1 - (normalizedData['WL2PTime'] / maximum_WL2))
TAU_WL2N = normalizedData['WL2N'] * (1 - (normalizedData['WL2NTime'] / maximum_WL2))

print(maximum_WL1)
print(maximum_WL2)

print('TAU(WL1P): \n', TAU_WL1P,  '\n The median of WL1P: ', TAU_WL1P.median(), '\n The mean of WL1P: ', TAU_WL1P.mean())
print('TAU(WL1N): \n', TAU_WL1N,  '\n The median of WL1N: ', TAU_WL1N.median(), '\n The mean of WL1N: ', TAU_WL1N.mean())

print('TAU(WL2P): \n', TAU_WL2P,  '\n The median of WL2P: ', TAU_WL2P.median(), '\n The mean of WL2P: ', TAU_WL2P.mean())
print('TAU(WL2N): \n', TAU_WL2N,  '\n The median of WL2N: ', TAU_WL2N.median(), '\n The mean of WL2N: ', TAU_WL2N.mean())

##### Metadata

In [ ]:
maximum_ME1 = pd.concat([normalizedData['ME1PTime'],normalizedData['ME1NTime']]).max()
maximum_ME2 = pd.concat([normalizedData['ME2PTime'],normalizedData['ME2NTime']]).max()
TAU_ME1P = normalizedData['ME1P'] * (1 - (normalizedData['ME1PTime'] / maximum_ME1))
TAU_ME1N = normalizedData['ME1N'] * (1 - (normalizedData['ME1NTime'] / maximum_ME1))
TAU_ME2P = normalizedData['ME2P'] * (1 - (normalizedData['ME2PTime'] / maximum_ME2))
TAU_ME2N = normalizedData['ME2N'] * (1 - (normalizedData['ME2NTime'] / maximum_ME2))

print(maximum_ME1)
print(maximum_ME2)

print('TAU(ME1P): \n', TAU_ME1P,  '\n The median of ME1P: ', TAU_ME1P.median(), '\n The mean of ME1P: ', TAU_ME1P.mean())
print('TAU(ME1N): \n', TAU_ME1N,  '\n The median of ME1N: ', TAU_ME1N.median(), '\n The mean of ME1N: ', TAU_ME1N.mean())

print('TAU(ME2P): \n', TAU_ME2P,  '\n The median of ME2P: ', TAU_ME2P.median(), '\n The mean of ME2P: ', TAU_ME2P.mean())
print('TAU(ME2N): \n', TAU_ME2N,  '\n The median of ME2N: ', TAU_ME2N.median(), '\n The mean of ME2N: ', TAU_ME2N.mean())

##### API Gateway

In [ ]:
maximum_AG1 = pd.concat([normalizedData['AG1PTime'],normalizedData['AG1NTime']]).max()
maximum_AG2 = pd.concat([normalizedData['AG2PTime'],normalizedData['AG2NTime']]).max()
TAU_AG1P = normalizedData['AG1P'] * (1 - (normalizedData['AG1PTime'] / maximum_AG1))
TAU_AG1N = normalizedData['AG1N'] * (1 - (normalizedData['AG1NTime'] / maximum_AG1))
TAU_AG2P = normalizedData['AG2P'] * (1 - (normalizedData['AG2PTime'] / maximum_AG2))
TAU_AG2N = normalizedData['AG2N'] * (1 - (normalizedData['AG2NTime'] / maximum_AG2))

print(maximum_AG1)
print(maximum_AG2)

print('TAU(AG1P): \n', TAU_AG1P,  '\n The median of AG1P: ', TAU_AG1P.median(), '\n The mean of AG1P: ', TAU_AG1P.mean())
print('TAU(AG1N): \n', TAU_AG1N,  '\n The median of AG1N: ', TAU_AG1N.median(), '\n The mean of AG1N: ', TAU_AG1N.mean())

print('TAU(AG2P): \n', TAU_AG2P,  '\n The median of AG2P: ', TAU_AG2P.median(), '\n The mean of AG2P: ', TAU_AG2P.mean())
print('TAU(AG2N): \n', TAU_AG2N,  '\n The median of AG2N: ', TAU_AG2N.median(), '\n The mean of AG2N: ', TAU_AG2N.mean())

##### Request Bundle

In [ ]:
maximum_RB1 = pd.concat([normalizedData['RB1PTime'],normalizedData['RB1NTime']]).max()
maximum_RB2 = pd.concat([normalizedData['RB2PTime'],normalizedData['RB2NTime']]).max()
TAU_RB1P = normalizedData['RB1P'] * (1 - (normalizedData['RB1PTime'] / maximum_RB1))
TAU_RB1N = normalizedData['RB1N'] * (1 - (normalizedData['RB1NTime'] / maximum_RB1))
TAU_RB2P = normalizedData['RB2P'] * (1 - (normalizedData['RB2PTime'] / maximum_RB2))
TAU_RB2N = normalizedData['RB2N'] * (1 - (normalizedData['RB2NTime'] / maximum_RB2))

print(maximum_RB1)
print(maximum_RB2)

print('TAU(RB1P): \n', TAU_RB1P,  '\n The median of RB1P: ', TAU_RB1P.median(), '\n The mean of RB1P: ', TAU_RB1P.mean())
print('TAU(RB1N): \n', TAU_RB1N,  '\n The median of RB1N: ', TAU_RB1N.median(), '\n The mean of RB1N: ', TAU_RB1N.mean())

print('TAU(RB2P): \n', TAU_RB2P,  '\n The median of RB2P: ', TAU_RB2P.median(), '\n The mean of RB2P: ', TAU_RB2P.mean())
print('TAU(RB2N): \n', TAU_RB2N,  '\n The median of RB2N: ', TAU_RB2N.median(), '\n The mean of RB2N: ', TAU_RB2N.mean())

**All boxplots together:**

In [ ]:
TAU_all = pd.concat([TAU_ER1P, TAU_ER1N, TAU_ER2P, TAU_ER2N, TAU_PT1P, TAU_PT1N, TAU_PT2P, TAU_PT2N, TAU_WL1P, TAU_WL1N, TAU_WL2P, TAU_WL2N,
                     TAU_ME1P, TAU_ME1N, TAU_ME2P, TAU_ME2N, TAU_RB1P, TAU_RB1N, TAU_RB2P, TAU_RB2N, TAU_AG1P, TAU_AG1N, TAU_AG2P, TAU_AG2N], axis=1)
TAU_all.columns = ['ER1P', 'ER1N', 'ER2P', 'ER2N', 'PT1P', 'PT1N', 'PT2P', 'PT2N', 'WL1P', 'WL1N', 'WL2P', 'WL2N',
                   'ME1P', 'ME1N', 'ME2P', 'ME2N', 'RB1P', 'RB1N', 'RB2P', 'RB2N', 'AG1P', 'AG1N', 'AG2P', 'AG2N']

ax, bp = TAU_all.boxplot(grid=False, medianprops=medianprops,
                         patch_artist=True, return_type="both", figsize=(15, 3))

# color the pairs of boxes differently
for index, patch in enumerate(bp["boxes"]):
    color = "lightblue" if index % 2 == 0 else "orange"
    patch.set(facecolor=color)

ax.set_ylim(-0.05, 1.05)
ax.set_ylabel("TAU", fontsize=18, labelpad=2.0)

# vertical lines between groups
[ax.axvline(x, color='black', linestyle='--')
 for x in [4.5, 8.5, 12.5, 16.5, 20.5]]

# pattern names above the groups
plt.text(1.3, 1.08, 'Error Report', fontsize=13)
plt.text(5, 1.08, 'Parameter Tree', fontsize=13)
plt.text(9.6, 1.08, 'Wish List', fontsize=13)
plt.text(12.8, 1.08, 'Metadata Element', fontsize=13)
plt.text(17.3, 1.08, 'Request Bundle', fontsize=13)
plt.text(21, 1.08, 'API Gateway', fontsize=13)

plt.plot()
# plt.savefig("tau-boxplots.pdf", dpi=150)

#### Results Overwiew

In [ ]:
# Initialize overview table
overViewTable = pd.DataFrame(columns = ['Mean TAU', 'Median TAU', 'Mean Duration (s)', 'Correct answers (%)'])

# Add data to table
overViewTable.loc['ER1P'] = pd.Series({'Mean TAU': TAU_ER1P.mean(), 'Median TAU': TAU_ER1P.median(), 'Mean Duration (s)': normalizedData['ER1PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['ER1P'].value_counts()[1] / normalizedData['ER1P'].count()}).round(4)
overViewTable.loc['ER1N'] = pd.Series({'Mean TAU': TAU_ER1N.mean(), 'Median TAU': TAU_ER1N.median(), 'Mean Duration (s)': normalizedData['ER1NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['ER1N'].value_counts()[1] / normalizedData['ER1N'].count()}).round(4)
overViewTable.loc['ER2P'] = pd.Series({'Mean TAU': TAU_ER2P.mean(), 'Median TAU': TAU_ER2P.median(), 'Mean Duration (s)': normalizedData['ER2PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['ER2P'].value_counts()[1] / normalizedData['ER2P'].count()}).round(4)
overViewTable.loc['ER2N'] = pd.Series({'Mean TAU': TAU_ER2N.mean(), 'Median TAU': TAU_ER2N.median(), 'Mean Duration (s)': normalizedData['ER2NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['ER2N'].value_counts()[1] / normalizedData['ER2N'].count()}).round(4)

overViewTable.loc['-'] = '-'

overViewTable.loc['PT1P'] = pd.Series({'Mean TAU': TAU_PT1P.mean(), 'Median TAU': TAU_PT1P.median(), 'Mean Duration (s)': normalizedData['PT1PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['PT1P'].value_counts()[1] / normalizedData['PT1P'].count()}).round(4)
overViewTable.loc['PT1N'] = pd.Series({'Mean TAU': TAU_PT1N.mean(), 'Median TAU': TAU_PT1N.median(), 'Mean Duration (s)': normalizedData['PT1NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['PT1N'].value_counts()[1] / normalizedData['PT1N'].count()}).round(4)
overViewTable.loc['PT2P'] = pd.Series({'Mean TAU': TAU_PT2P.mean(), 'Median TAU': TAU_PT2P.median(), 'Mean Duration (s)': normalizedData['PT2PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['PT2P'].value_counts()[1] / normalizedData['PT2P'].count()}).round(4)
overViewTable.loc['PT2N'] = pd.Series({'Mean TAU': TAU_PT2N.mean(), 'Median TAU': TAU_PT2N.median(), 'Mean Duration (s)': normalizedData['PT2NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['PT2N'].value_counts()[1] / normalizedData['PT2N'].count()}).round(4)

overViewTable.loc['- '] = '-'

overViewTable.loc['WL1P'] = pd.Series({'Mean TAU': TAU_WL1P.mean(), 'Median TAU': TAU_WL1P.median(), 'Mean Duration (s)': normalizedData['WL1PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['WL1P'].value_counts()[1] / normalizedData['WL1P'].count()}).round(4)
overViewTable.loc['WL1N'] = pd.Series({'Mean TAU': TAU_WL1N.mean(), 'Median TAU': TAU_WL1N.median(), 'Mean Duration (s)': normalizedData['WL1NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['WL1N'].value_counts()[1] / normalizedData['WL1N'].count()}).round(4)
overViewTable.loc['WL2P'] = pd.Series({'Mean TAU': TAU_WL2P.mean(), 'Median TAU': TAU_WL2P.median(), 'Mean Duration (s)': normalizedData['WL2PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['WL2P'].value_counts()[1] / normalizedData['WL2P'].count()}).round(4)
overViewTable.loc['WL2N'] = pd.Series({'Mean TAU': TAU_WL2N.mean(), 'Median TAU': TAU_WL2N.median(), 'Mean Duration (s)': normalizedData['WL2NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['WL2N'].value_counts()[1] / normalizedData['WL2N'].count()}).round(4)

overViewTable.loc[' -  '] = '-'

overViewTable.loc['ME1P'] = pd.Series({'Mean TAU': TAU_ME1P.mean(), 'Median TAU': TAU_ME1P.median(), 'Mean Duration (s)': normalizedData['ME1PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['ME1P'].value_counts()[1] / normalizedData['ME1P'].count()}).round(4)
overViewTable.loc['ME1N'] = pd.Series({'Mean TAU': TAU_ME1N.mean(), 'Median TAU': TAU_ME1N.median(), 'Mean Duration (s)': normalizedData['ME1NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['ME1N'].value_counts()[1] / normalizedData['ME1N'].count()}).round(4)
overViewTable.loc['ME2P'] = pd.Series({'Mean TAU': TAU_ME2P.mean(), 'Median TAU': TAU_ME2P.median(), 'Mean Duration (s)': normalizedData['ME2PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['ME2P'].value_counts()[1] / normalizedData['ME2P'].count()}).round(4)
overViewTable.loc['ME2N'] = pd.Series({'Mean TAU': TAU_ME2N.mean(), 'Median TAU': TAU_ME2N.median(), 'Mean Duration (s)': normalizedData['ME2NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['ME2N'].value_counts()[1] / normalizedData['ME2N'].count()}).round(4)

overViewTable.loc[' - '] = '-'

overViewTable.loc['AG1P'] = pd.Series({'Mean TAU': TAU_AG1P.mean(), 'Median TAU': TAU_AG1P.median(), 'Mean Duration (s)': normalizedData['AG1PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['AG1P'].value_counts()[1] / normalizedData['AG1P'].count()}).round(4)
overViewTable.loc['AG1N'] = pd.Series({'Mean TAU': TAU_AG1N.mean(), 'Median TAU': TAU_AG1N.median(), 'Mean Duration (s)': normalizedData['AG1NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['AG1N'].value_counts()[1] / normalizedData['AG1N'].count()}).round(4)
overViewTable.loc['AG2P'] = pd.Series({'Mean TAU': TAU_AG2P.mean(), 'Median TAU': TAU_AG2P.median(), 'Mean Duration (s)': normalizedData['AG2PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['AG2P'].value_counts()[1] / normalizedData['AG2P'].count()}).round(4)
overViewTable.loc['AG2N'] = pd.Series({'Mean TAU': TAU_AG2N.mean(), 'Median TAU': TAU_AG2N.median(), 'Mean Duration (s)': normalizedData['AG2NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['AG2N'].value_counts()[1] / normalizedData['AG2N'].count()}).round(4)

overViewTable.loc['  -  '] = '-'

overViewTable.loc['RB1P'] = pd.Series({'Mean TAU': TAU_RB1P.mean(), 'Median TAU': TAU_RB1P.median(), 'Mean Duration (s)': normalizedData['RB1PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['RB1P'].value_counts()[1] / normalizedData['RB1P'].count()}).round(4)
overViewTable.loc['RB1N'] = pd.Series({'Mean TAU': TAU_RB1N.mean(), 'Median TAU': TAU_RB1N.median(), 'Mean Duration (s)': normalizedData['RB1NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['RB1N'].value_counts()[1] / normalizedData['RB1N'].count()}).round(4)
overViewTable.loc['RB2P'] = pd.Series({'Mean TAU': TAU_RB2P.mean(), 'Median TAU': TAU_RB2P.median(), 'Mean Duration (s)': normalizedData['RB2PTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['RB2P'].value_counts()[1] / normalizedData['RB2P'].count()}).round(4)
overViewTable.loc['RB2N'] = pd.Series({'Mean TAU': TAU_RB2N.mean(), 'Median TAU': TAU_RB2N.median(), 'Mean Duration (s)': normalizedData['RB2NTime'].mean().round(2), 'Correct answers (%)': 100 * normalizedData['RB2N'].value_counts()[1] / normalizedData['RB2N'].count()}).round(4)

overViewTable

#### Check for Normality with Shapiro-Wilk's W-test

In [ ]:
L = [
    ['Error Report 1 Task (P)', TAU_ER1P], ['Error Report 1 Task (N)', TAU_ER1N], ['Error Report 2 Task (P)', TAU_ER2P], ['Error Report 2 Task (N)', TAU_ER2N],
    ['Parameter Tree 1 Task (P)', TAU_PT1P], ['Parameter Tree 1 Task (N)', TAU_PT1N], ['Parameter Tree 2 Task (P)', TAU_PT2P], ['Parameter Tree 2 Task (N)', TAU_PT2N],
    ['Wish List 1 Task (P)', TAU_WL1P], ['Wish List 1 Task (N)', TAU_WL1N], ['Wish List 2 Task (P)', TAU_WL2P], ['Wish List 2 Task (N)', TAU_WL2N],
    ['Metadata 1 Task (P)', TAU_ME1P], ['Metadata 1 Task (N)', TAU_ME1N], ['Metadata 2 Task (P)', TAU_ME2P], ['Metadata 2 Task (N)', TAU_ME2N],
    ['API Gateway 1 Task (P)', TAU_AG1P], ['API Gateway 1 Task (N)', TAU_AG1N], ['API Gateway 2 Task (P)', TAU_AG2P], ['API Gateway 2 Task (N)', TAU_AG2N],
    ['Request Bundle 1 Task (P)', TAU_RB1P], ['Request Bundle 1 Task (N)', TAU_RB1N], ['Request Bundle 2 Task (P)', TAU_RB2P], ['Request Bundle 2 Task (N)', TAU_RB2N]    
     ]

for task in L :
    w_normality, p_normality = stats.shapiro(task[1].dropna())

    if p_normality < 0.05:
        L
        ##print("The data of", task[0],"(p-value = %.6f" % p_normality, ") is NOT normally distributed!")
    else:
        print("The data of", task[0],"(p-value = %.6f" % p_normality, ") IS normally distributed!")

Result: Parameter Tree 2 Task (N) and Metadata 2 Task (P) are the only two datasets that are normally distributed.

#### The Mann-Whitney U Test
The concatenation of TAU_XX1P and TAU_XX2P are tested together against the concatenation of TAU_XX1N and TAU_XX2N.

In [ ]:
# Signifance level for the upcoming tests
alpha = 0.05

In [ ]:
u_value_ER, p_value_ER = stats.mannwhitneyu(pd.concat([TAU_ER1P, TAU_ER2P]).dropna(), pd.concat([TAU_ER1N, TAU_ER2N]).dropna(), alternative="greater")

print("Error Report p-value: %.6f" % p_value_ER, "U: %.2f" % u_value_ER)

if p_value_ER < alpha:   
    print("Null hypothesis (H₀) for Error Report Pattern REJECTED!")
else:
    print("Null hypothesis (H₀) for Error Report Pattern cannot be rejected!")

In [ ]:
u_value_PT, p_value_PT = stats.mannwhitneyu(pd.concat([TAU_PT1P, TAU_PT2P]).dropna(), pd.concat([TAU_PT1N, TAU_PT2N]).dropna(), alternative="greater")

print("Parameter Tree p-value: %.6f" % p_value_PT, "U: %.2f" % u_value_PT)

if p_value_PT < alpha:   
    print("Null hypothesis (H₀) for Parameter Tree Pattern REJECTED!")
else:
    print("Null hypothesis (H₀) for Parameter Tree Pattern cannot be rejected!")

In [ ]:
u_value_WL, p_value_WL = stats.mannwhitneyu(pd.concat([TAU_WL1P, TAU_WL2P]).dropna(), pd.concat([TAU_WL1N, TAU_WL2N]).dropna(), alternative="greater")

print("Wish List p-value: %.6f" % p_value_WL, "U: %.2f" % u_value_WL)

if p_value_WL < alpha:   
    print("Null hypothesis (H₀) for Wish List Pattern REJECTED!")
else:
    print("Null hypothesis (H₀) for Wish List Pattern cannot be rejected!")

In [ ]:
u_value_ME, p_value_ME = stats.mannwhitneyu(pd.concat([TAU_ME1P, TAU_ME2P]).dropna(), pd.concat([TAU_ME1N, TAU_ME2N]).dropna(), alternative="greater")

print("Metadata p-value: %.6f" % p_value_ME, "U: %.2f" % u_value_ME)

if p_value_ME < alpha:   
    print("Null hypothesis (H₀) for Metadata Pattern REJECTED!")
else:
    print("Null hypothesis (H₀) for Metadata Pattern cannot be rejected!")

In [ ]:
u_value_AG, p_value_AG = stats.mannwhitneyu(pd.concat([TAU_AG1P, TAU_AG2P]).dropna(), pd.concat([TAU_AG1N, TAU_AG2N]).dropna(), alternative="greater")

print("API Gateway p-value: %.6f" % p_value_AG, "U: %.2f" % u_value_AG)

if p_value_AG < alpha:   
    print("Null hypothesis (H₀) for API Gateway Pattern REJECTED!")
else:
    print("Null hypothesis (H₀) for API Gateway Pattern cannot be rejected!")

In [ ]:
u_value_RB, p_value_RB = stats.mannwhitneyu(pd.concat([TAU_RB1P, TAU_RB2P]).dropna(), pd.concat([TAU_RB1N, TAU_RB2N]).dropna(), alternative="greater")

print("Request Bundle p-value: %.6f" % p_value_RB, "U: %.2f" % u_value_RB)

if p_value_RB < alpha:   
    print("Null hypothesis (H₀) for Request Bundle Pattern REJECTED!")
else:
    print("Null hypothesis (H₀) for Request Bundle Pattern cannot be rejected!")

#### Adjusting p-values with Holm-Bonferroni method

*Method used: https://en.wikipedia.org/wiki/Holm%E2%80%93Bonferroni_method*

*Documentation for `statsmodels.stats.multipletests()`: https://www.statsmodels.org/dev/generated/statsmodels.stats.multitest.multipletests.html#statsmodels.stats.multitest.multipletests*

In [ ]:
reject, pvals_corrected, alphacSidak, aphacBonf = stm.stats.multipletests((p_value_ER, p_value_PT, p_value_WL, p_value_ME, p_value_AG, p_value_RB), alpha = alpha, method = 'holm',  is_sorted = False, returnsorted = False)

print("Holm-Bonferroni p correction completed. \nThe Null hypotheses (H₀) for which of the following patterns can be REJECTED (with alpha: %.2f)?"% alpha, "\n")
print(  "Error Report:", reject[0], "(corrected p-value: %.6f)" % pvals_corrected[0],
        "\nParameter Tree:", reject[1], "(corrected p-value: %.6f)" % pvals_corrected[1],
        "\nWish List:", reject[2], "(corrected p-value: %.6f)" % pvals_corrected[2],
        "\nMetadata:", reject[3], "(corrected p-value: %.6f)" % pvals_corrected[3],
        "\nAPI Gateway:", reject[4], "(corrected p-value: %.6f)" % pvals_corrected[4],
        "\nRequest Bundle:", reject[5], "(corrected p-value: %.6f)" % pvals_corrected[5])

#### Effect Size with Cohen's d calculation

*Info: https://en.wikipedia.org/wiki/Effect_size#Cohen's_d*

##### Error Report

In [ ]:
# Calculate length of TAU groups (can vary because of dropped datapoints)
length_ERP = len(pd.concat([TAU_ER1P, TAU_ER2P]).dropna())
length_ERN = len(pd.concat([TAU_ER1N, TAU_ER2N]).dropna())

# Difference of the means
diff_mean_ER = pd.concat([TAU_ER1P, TAU_ER2P]).dropna().mean() - pd.concat([TAU_ER1N, TAU_ER2N]).dropna().mean()

# Calculate standard deviation for both TAU groups
s_deviation_ERP = ((pd.concat([TAU_ER1P, TAU_ER2P]).dropna()).std())
s_deviation_ERN = ((pd.concat([TAU_ER1N, TAU_ER2N]).dropna()).std())

# Calculate pooled standard deviation
pooled_std_ER = np.sqrt(((length_ERP - 1)* s_deviation_ERP * s_deviation_ERP + (length_ERN - 1) * s_deviation_ERN * s_deviation_ERN) / (length_ERP + length_ERN - 2))

# Calculate Cohen's d
cohens_d_ER = diff_mean_ER / pooled_std_ER

print("Pooled standard deviation for Error Report: %.4f" % pooled_std_ER)
print("Cohen's d for Error Report: %.2f" % cohens_d_ER)

##### Parameter Tree

In [ ]:
# Calculate length of TAU groups (can vary because of dropped datapoints)
length_PTP = len(pd.concat([TAU_PT1P, TAU_PT2P]).dropna())
length_PTN = len(pd.concat([TAU_PT1N, TAU_PT2N]).dropna())

# Difference of the means
diff_mean_PT = pd.concat([TAU_PT1P, TAU_PT2P]).dropna().mean() - pd.concat([TAU_PT1N, TAU_PT2N]).dropna().mean()

# Calculate standard deviation for both TAU groups
s_deviation_PTP = ((pd.concat([TAU_PT1P, TAU_PT2P]).dropna()).std())
s_deviation_PTN = ((pd.concat([TAU_PT1N, TAU_PT2N]).dropna()).std())

# Calculate pooled standard deviation
pooled_std_PT = np.sqrt(((length_PTP - 1)* s_deviation_PTP * s_deviation_PTP + (length_PTN - 1) * s_deviation_PTN * s_deviation_PTN) / (length_PTP + length_PTN - 2))

# Calculate Cohen's d
cohens_d_PT = diff_mean_PT / pooled_std_PT

print("Pooled standard deviation for Parameter Tree: %.4f" % pooled_std_PT)
print("Cohen's d for Parameter Tree: %.2f" % cohens_d_PT)

##### Wish List

In [ ]:
# Calculate length of TAU groups (can vary because of dropped datapoints)
length_WLP = len(pd.concat([TAU_WL1P, TAU_WL2P]).dropna())
length_WLN = len(pd.concat([TAU_WL1N, TAU_WL2N]).dropna())

# Difference of the means
diff_mean_WL = pd.concat([TAU_WL1P, TAU_WL2P]).dropna().mean() - pd.concat([TAU_WL1N, TAU_WL2N]).dropna().mean()

# Calculate standard deviation for both TAU groups
s_deviation_WLP = ((pd.concat([TAU_WL1P, TAU_WL2P]).dropna()).std())
s_deviation_WLN = ((pd.concat([TAU_WL1N, TAU_WL2N]).dropna()).std())

# Calculate pooled standard deviation
pooled_std_WL = np.sqrt(((length_WLP - 1)* s_deviation_WLP * s_deviation_WLP + (length_WLN - 1) * s_deviation_WLN * s_deviation_WLN) / (length_WLP + length_WLN - 2))

# Calculate Cohen's d
cohens_d_WL = diff_mean_WL / pooled_std_WL

print("Pooled standard deviation for Wish List: %.4f" % pooled_std_WL)
print("Cohen's d for Wish List: %.2f" % cohens_d_WL)

##### Metadata

In [ ]:
# Calculate length of TAU groups (can vary because of dropped datapoints)
length_MEP = len(pd.concat([TAU_ME1P, TAU_ME2P]).dropna())
length_MEN = len(pd.concat([TAU_ME1N, TAU_ME2N]).dropna())

# Difference of the means
diff_mean_ME = pd.concat([TAU_ME1P, TAU_ME2P]).dropna().mean() - pd.concat([TAU_ME1N, TAU_ME2N]).dropna().mean()

# Calculate standard deviation for both TAU groups
s_deviation_MEP = ((pd.concat([TAU_ME1P, TAU_ME2P]).dropna()).std())
s_deviation_MEN = ((pd.concat([TAU_ME1N, TAU_ME2N]).dropna()).std())

# Calculate pooled standard deviation
pooled_std_ME = np.sqrt(((length_MEP - 1)* s_deviation_MEP * s_deviation_MEP + (length_MEN - 1) * s_deviation_MEN * s_deviation_MEN) / (length_MEP + length_MEN - 2))

# Calculate Cohen's d
cohens_d_ME = diff_mean_ME / pooled_std_ME

print("Pooled standard deviation for Metadata: %.4f" % pooled_std_ME)
print("Cohen's d for Metadata: %.2f" % cohens_d_ME)

##### API Gateway

In [ ]:
# Calculate length of TAU groups (can vary because of dropped datapoints)
length_AGP = len(pd.concat([TAU_AG1P, TAU_AG2P]).dropna())
length_AGN = len(pd.concat([TAU_AG1N, TAU_AG2N]).dropna())

# Difference of the means
diff_mean_AG = pd.concat([TAU_AG1P, TAU_AG2P]).dropna().mean() - pd.concat([TAU_AG1N, TAU_AG2N]).dropna().mean()

# Calculate standard deviation for both TAU groups
s_deviation_AGP = ((pd.concat([TAU_AG1P, TAU_AG2P]).dropna()).std())
s_deviation_AGN = ((pd.concat([TAU_AG1N, TAU_AG2N]).dropna()).std())

# Calculate pooled standard deviation
pooled_std_AG = np.sqrt(((length_AGP - 1)* s_deviation_AGP * s_deviation_AGP + (length_AGN - 1) * s_deviation_AGN * s_deviation_AGN) / (length_AGP + length_AGN - 2))

# Calculate Cohen's d
cohens_d_AG = diff_mean_AG / pooled_std_AG

print("Pooled standard deviation for API Gateway: %.4f" % pooled_std_AG)
print("Cohen's d for API Gateway: %.2f" % cohens_d_AG)

##### Request Bundle

Null hypothesis for 'Request Bundle' could not be rejected, thus the effect size calculation in the following is NOT needed

In [ ]:
# Calculate length of TAU groups (can vary because of dropped datapoints)
length_RBP = len(pd.concat([TAU_RB1P, TAU_RB2P]).dropna())
length_RBN = len(pd.concat([TAU_RB1N, TAU_RB2N]).dropna())

# Difference of the means
diff_mean_RB = pd.concat([TAU_RB1P, TAU_RB2P]).dropna().mean() - pd.concat([TAU_RB1N, TAU_RB2N]).dropna().mean()

# Calculate standard deviation for both TAU groups
s_deviation_RBP = ((pd.concat([TAU_RB1P, TAU_RB2P]).dropna()).std())
s_deviation_RBN = ((pd.concat([TAU_RB1N, TAU_RB2N]).dropna()).std())

# Calculate pooled standard deviation
pooled_std_RB = np.sqrt(((length_RBP - 1)* s_deviation_RBP * s_deviation_RBP + (length_RBN - 1) * s_deviation_RBN * s_deviation_RBN) / (length_RBP + length_RBN - 2))

# Calculate Cohen's d
cohens_d_RB = diff_mean_RB / pooled_std_RB

print("Pooled standard deviation for Request Bundle: %.4f" % pooled_std_RB)
print("Cohen's d for Request Bundle: %.2f" % cohens_d_RB)

## Impact of Demographics (RQ2)

Kendall's TAU: https://en.wikipedia.org/wiki/Kendall_rank_correlation_coefficient

Look for possible correlations between the demographic data and TAU (exploratory RQ2)

### Calculate Mean TAU per Participant and Version

In [ ]:
dems = demographics.drop(columns=["randnumber", "DemAcInd", "DemBackground", "DemExperience[SQ001]",
                                  "DemExperience[SQ002]", "DemExperience[SQ003]"])
dems["meanTAU_P"] = pd.DataFrame([TAU_ER1P, TAU_ER2P, TAU_PT1P, TAU_PT2P, TAU_WL1P,
                                 TAU_WL2P, TAU_ME1P, TAU_ME2P, TAU_AG1P, TAU_AG2P, TAU_RB1P, TAU_RB2P]).agg("mean")
dems["meanTAU_N"] = pd.DataFrame([TAU_ER1N, TAU_ER2N, TAU_PT1N, TAU_PT2N, TAU_WL1N,
                                 TAU_WL2N, TAU_ME1N, TAU_ME2N, TAU_AG1N, TAU_AG2N, TAU_RB1N, TAU_RB2N]).agg("mean")

### Correlation Matrix for Visual Exploration

In [ ]:
# correlation coefficients visualization
corr = dems.corr(method="kendall")
corr.style.background_gradient(cmap="PuOr_r").format(precision=3)

In [ ]:
# p-values visualization

def kendall_pval(x, y):
    return stats.kendalltau(x, y)[1]


corr = dems.corr(method=kendall_pval)
corr.style.background_gradient(cmap="PuOr_r").format(precision=8)

### Linear Regression Models to Predit TAU per Version

In [ ]:
# predict TAU for pattern tasks
x = dems[["DemYearsOE", "isStudent", "isIndustry",
          "DemExperience[patterns]", "DemExperience[APIs]", "DemExperience[MAPs]"]].astype(float)
x = stm.add_constant(x)
y = dems["meanTAU_P"].astype(float)
lm = stm.OLS(y, x)
res = lm.fit()
print(res.summary())

In [ ]:
# predict TAU for non-pattern tasks
y = dems["meanTAU_N"].astype(float)
lm = stm.OLS(y, x)
res = lm.fit()
print(res.summary())